# Heterogeneous Regression and Classification: Constrained BO

回帰の強度・導電率と二値分類の合否を別モデルとして学習し、名前付き出力を `HeterogeneousModel` に統合します。強度最大化と導電率・合否制約から獲得関数を作り、候補提案と真値評価まで実行します。

このNotebookは異種**出力**モデルを扱います。タスクごとの入力特徴量が異なる `HeterogeneousMTGP` とは別です。


In [ ]:
import torch
from botorch.fit import fit_gpytorch_mll
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.acquisition.composition import make_mixed_constrained_qei_acquisition
from robotorchan.benchmarks.heterogeneous_synthetic import evaluate_truth, observe
from robotorchan.models.classification import BinarySingleTaskGPClassifier
from robotorchan.models.heterogeneous import HeterogeneousModel
from robotorchan.models.standard.single_task import SingleTaskGP
from robotorchan.optim.dispatch import optimize_acqf
from robotorchan.semantics import (
    ClassificationConstraint,
    ContinuousConstraint,
    ProblemSemantics,
    RegressionObjective,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(1212)


## 1. 観測データとモデルの学習

回帰GPはMLLで学習し、二値分類GPは変分目的関数をAdamで最適化します。


In [ ]:
generator = torch.Generator().manual_seed(1212)
train_X = torch.rand(24, 3, generator=generator)
observed = observe(train_X, seed=2212)
train_Y = torch.stack((observed.strength, observed.conductivity), dim=-1)

regression = SingleTaskGP(
    train_X,
    train_Y,
    train_Yvar=torch.full_like(train_Y, 0.02**2),
)
fit_gpytorch_mll(regression.make_mll())
regression.eval()

classifier = BinarySingleTaskGPClassifier(
    train_X,
    observed.passed,
    inducing_points=8,
)
classifier.train()
classifier.likelihood.train()
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.03)
for _ in range(12):
    optimizer.zero_grad()
    output = classifier.model(train_X)
    loss = -classifier.make_mll()(output, observed.passed.to(dtype=train_X.dtype))
    loss.backward()
    optimizer.step()
classifier.eval()
classifier.likelihood.eval()


## 2. 出力の意味付けと制約付き獲得関数

強度を最大化し、導電率の下限と合格クラスの確率を組み合わせます。分類のlatent GP平均を合格確率として扱いません。


In [ ]:
model = HeterogeneousModel(
    regression,
    classifier,
    output_names=["strength", "conductivity", "pass"],
)
semantics = ProblemSemantics(
    objectives=(RegressionObjective("strength"),),
    constraints=(
        ContinuousConstraint("conductivity", threshold=0.8),
        ClassificationConstraint("pass", feasible_class=1),
    ),
)
acquisition = make_mixed_constrained_qei_acquisition(
    model,
    semantics,
    best_f=train_Y[:, 0].max().item(),
    sampler=SobolQMCNormalSampler(torch.Size([16]), seed=1212),
)


## 3. 候補の提案と実際の評価

非微分的な構成も扱える有限Sobol探索を使います。optimizer APIの互換性そのものは別の監査フローで扱います。


In [ ]:
bounds = torch.stack((torch.zeros(3), torch.ones(3)))
candidate, score = optimize_acqf(
    acquisition,
    bounds,
    q=1,
    optimizer="sobol",
    raw_samples=32,
    seed=1213,
)
with torch.no_grad():
    strength, conductivity, pass_probability = evaluate_truth(candidate)
    predicted_pass_probability = model.entry_predict_proba(1, candidate)[..., 1]

assert candidate.shape == torch.Size([1, 3])
assert torch.isfinite(score).all()
assert torch.isfinite(strength).all()
assert torch.isfinite(conductivity).all()
assert torch.isfinite(pass_probability).all()
print("candidate:", candidate)
print("acquisition score:", score)
print("strength:", strength)
print("conductivity:", conductivity)
print("true pass probability:", pass_probability)
print("predicted pass probability:", predicted_pass_probability)


## 4. 観測とモデル予測の区別

`evaluate_truth` の合格確率は合成問題の真値であり、実験で観測される二値ラベルとは異なります。分類器を再学習する場合は、新しい観測ラベルを取得してから学習データへ追加してください。ここでは候補の評価までを示します。
